In [ ]:
import polars as pl
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

lf = pl.scan_parquet(
    "../data/processed/taxi_2025-01_features.parquet"
)  # lazy scan - loads on collect()
df = lf.collect()

In [ ]:
# Features and target
categorical_cols = ["pickup_borough", "dropoff_borough"]
numeric_cols = ["hour", "day_of_week", "is_rush_hour", "is_airport_pickup"]
target = "duration_min"

In [7]:
mean_baseline = df["duration_min"].mean()
median_baseline = df["duration_min"].median()

df = df.with_columns(
    (pl.col("duration_min") - mean_baseline).alias("mean_baseline_error"),
    (pl.col("duration_min") - median_baseline).alias("median_baseline_error"),
)

In [8]:
print(f"Mean Baseline MAE: {df['mean_baseline_error'].abs().mean()}")
print(f"Median Baseline MAE: {df['median_baseline_error'].abs().mean()}")

Mean Baseline MAE: 7.800973951065578
Median Baseline MAE: 7.383806856589194


In [ ]:
# Data split
train = df.filter(pl.col("pickup_date") < pl.date(2025, 1, 25))
holdout = df.filter(pl.col("pickup_date") >= pl.date(2025, 1, 25))

X_train = train.select(categorical_cols + numeric_cols).to_pandas()
y_train = train[target].to_pandas()
X_holdout = holdout.select(categorical_cols + numeric_cols).to_pandas()
y_holdout = holdout[target].to_pandas()

In [ ]:
# Preprocessor
preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(drop=["manhattan"]), categorical_cols),
    ("num", "passthrough", numeric_cols),
])

In [ ]:
# Pipeline definition
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

pipeline.fit(X_train, y_train)

In [ ]:
train_mae = mean_absolute_error(y_train, pipeline.predict(X_train))
holdout_mae = mean_absolute_error(y_holdout, pipeline.predict(X_holdout))
print(f"Train MAE: {train_mae:.4f}")
print(f"Holdout MAE: {holdout_mae:.4f}")